# 20 – Teams Bot (Adaptive Cards & Webhook)

Tests the Microsoft Teams integration layer:
- **Adaptive Card builders**: response, HITL approval, error, welcome, thinking
- **Activity routing**: message, invoke (approve/reject), conversationUpdate
- **HMAC verification**: signature validation
- **Webhook endpoint**: via FastAPI TestClient

In [ ]:
import sys, os, json
sys.path.insert(0, os.path.abspath('../src'))
os.environ['ENABLE_MOCK'] = 'true'

## 1. build_response_card

In [ ]:
from teams.cards import build_response_card

result = {
    'final_summary': 'Retention GRR is 92.5%, above the 85% threshold. All metrics healthy.',
    'confidence': 0.92,
    'anomalies': [],
    'auto_tickets': [],
    'errors': [],
}

card = build_response_card(result)
print('Envelope type   :', card.get('type'))
print('Attachment count:', len(card['attachments']))

content = card['attachments'][0]['content']
print('Card type       :', content['type'])
print('Card version    :', content['version'])
print('Body blocks     :', len(content['body']))
for block in content['body']:
    print(f"  [{block['type']}] {str(block.get('text', ''))[:60]}")

## 2. build_response_card — with anomalies and tickets

In [ ]:
result_with_anomalies = {
    'final_summary': 'GRR dropped to 78%. Tickets created.',
    'confidence': 0.88,
    'anomalies': [
        'retention: GRR 78.0% is below threshold 85.0%',
        'retention: 87 at-risk accounts exceeds limit 30',
    ],
    'auto_tickets': ['DGC-101: retention GRR anomaly'],
    'errors': [],
}

card = build_response_card(result_with_anomalies)
content = card['attachments'][0]['content']

warning_blocks = [b for b in content['body'] if b.get('color') == 'Warning']
print(f'Warning blocks (anomalies): {len(warning_blocks)}')
for b in warning_blocks:
    print(f"  {b['text'][:60]}")

## 3. build_hitl_card — approval prompt

In [ ]:
from teams.cards import build_hitl_card

pending = {
    'type': 'create_tickets',
    'message': '2 anomalies detected. Create Jira tickets?',
    'count': 2,
    'anomalies': ['GRR 78% below threshold', '87 at-risk accounts'],
}

card = build_hitl_card(pending, thread_id='thread-xyz', query='show retention metrics')
content = card['attachments'][0]['content']

print('Actions:')
for action in content.get('actions', []):
    print(f"  [{action['type']}] {action['title']}  data={action['data']}")

## 4. build_error_card and build_welcome_card

In [ ]:
from teams.cards import build_error_card, build_welcome_card, build_thinking_card

error = build_error_card('Connection timeout to Databricks')
error_text = [b['text'] for b in error['attachments'][0]['content']['body']]
print('Error card body:', error_text)

welcome = build_welcome_card()
welcome_text = welcome['attachments'][0]['content']['body'][0]['text']
print('Welcome card title:', welcome_text)

thinking = build_thinking_card()
thinking_text = thinking['attachments'][0]['content']['body'][0]['text']
print('Thinking card:', thinking_text)

## 5. TeamsActivity models

In [ ]:
from teams.models import TeamsActivity, TeamsUser, TeamsConversation

# Simulate a Teams message payload
payload = {
    'type': 'message',
    'id': 'msg-001',
    'text': 'What is the GRR for retention?',
    'from': {'id': 'user-123', 'name': 'Alice'},
    'conversation': {'id': 'conv-456', 'isGroup': False, 'conversationType': 'personal'},
    'serviceUrl': 'https://smba.trafficmanager.net',
    'channelId': 'msteams',
}

activity = TeamsActivity.model_validate(payload)
print('type       :', activity.type)
print('text       :', activity.text)
print('from_.name :', activity.from_.name)
print('conv.id    :', activity.conversation.id)
print('channel    :', activity.channel_id)

## 6. Webhook endpoint — message activity

In [ ]:
from fastapi.testclient import TestClient
from api.app import app

client = TestClient(app, raise_server_exceptions=False)

payload = json.dumps({
    'type': 'message',
    'id': 'msg-001',
    'text': 'What is the retention GRR?',
    'from': {'id': 'user-1', 'name': 'Alice'},
    'conversation': {'id': 'conv-1'},
    'channelId': 'msteams',
})

resp = client.post(
    '/teams/webhook',
    content=payload,
    headers={'Content-Type': 'application/json'},
)

print('Status :', resp.status_code)
data = resp.json()
print('type   :', data.get('type'))
content = (data.get('attachments') or [{}])[0].get('content', {})
print('card type:', content.get('type'))
body_texts = [b.get('text', '')[:60] for b in content.get('body', [])]
for t in body_texts:
    print(f'  {t}')

## 7. Webhook endpoint — conversationUpdate (welcome)

In [ ]:
payload = json.dumps({
    'type': 'conversationUpdate',
    'id': 'event-001',
    'membersAdded': [{'id': 'bot-1', 'name': 'DataBot'}],
    'conversation': {'id': 'conv-2'},
    'channelId': 'msteams',
})

resp = client.post(
    '/teams/webhook',
    content=payload,
    headers={'Content-Type': 'application/json'},
)

print('Status:', resp.status_code)
data = resp.json()
content = (data.get('attachments') or [{}])[0].get('content', {})
print('Welcome card title:', content.get('body', [{}])[0].get('text', ''))

## 8. Invoke — approve tickets

In [ ]:
payload = json.dumps({
    'type': 'invoke',
    'id': 'invoke-001',
    'value': {
        'action': 'approve_tickets',
        'thread_id': 'conv-1',
        'query': 'show retention metrics',
    },
    'conversation': {'id': 'conv-1'},
    'channelId': 'msteams',
})

resp = client.post(
    '/teams/webhook',
    content=payload,
    headers={'Content-Type': 'application/json'},
)

print('Status:', resp.status_code)
data = resp.json()
content = (data.get('attachments') or [{}])[0].get('content', {})
print('Card type:', content.get('type'))
# Should be response card (approved)
for b in content.get('body', [])[:2]:
    print(f"  {b.get('text', '')[:60]}")